### 🎯 프로젝트 **목표**

- 이번 과제는 단순히 모델을 구현하는 데서 끝나는 것이 아니라, **기존 Baseline 모델의 한계를 스스로 파악하고 더 나은 방향으로 개선해보는 과정 자체**를 경험하는 데 목적이 있습니다.
- 다양한 실험과 비교를 통해, 하나의 정답을 찾기보다 **문제를 정의하고 가설을 세우고 검증하는 문제 해결 과정**을 직접 연습해보는 것이 중요합니다.
- 이번 과제는 여러분이 지금까지 배운 내용을 바탕으로 진행되지만, **필요하다면 배우지 않은 내용도 스스로 찾아보고 적용해보는 경험**까지 포함합니다.
- 성능 수치만 높이는 것이 아니라, **왜 이런 전략을 선택했는지, 어떤 실험이 효과가 있었는지, 어떤 실패를 통해 무엇을 배웠는지**까지 함께 정리해보며 한 단계 더 성장해보는 것을 목표로 합니다.
- 익숙한 방법만 반복하는 것이 아니라, 새로운 방법을 시도해보며 스스로 탐색하는 과정도 중요한 학습입니다.

## ⚠️ 과제 수행 필수 규칙

1. **사전 학습된(Pre-trained) 모델 활용 시**
    - **최소 2개 이상의 개선 전략 적용**
        - 단순 모델 변경 외에 추가적인 시도(전처리, Loss 조정, optimizer 변경, learning rate 조정 등)가 필수입니다.
        - **Baseline 대비 의미 있는 성능 비교 결과를 반드시 포함**해야 합니다.(단, 개선 폭이 크지 않더라도 실험 과정과 분석이 충분히 설명되면 평가에 반영됩니다.)
    - 왜 해당 모델을 선택하였으며 문제 해결에 적합하다고 생각했는지 명시하세요.
    - **Fine-tuning 방식**에서 어떤 부분을 고정하고, 어떤 부분을 학습했는지, 왜 그렇게 설정했는지 구체적으로 작성해주세요.
        1. **논리적 근거 제시**
            - 모델 선정 시 “좋아보여서”가 아닌 왜 현재 문제에 해당 모델이 적합했는지를 명시해주세요.
            - **AI 도구 사용은 가능**하지만, 왜 그 방법을 썼는지 본인의 언어로 설명할 수 있어야 합니다.
    - 단, **단순 적용만으로는 성능 개선으로 인정되지 않습니다.**
2. **재현성 확보**
    - 제출한 코드는 누구나 실행했을 때 동일한 결과가 나와야 하는 전체 파이프라인 형태여야 합니다.
3. **데일리 로그**
    - 매일의 진행 과정과 고민의 흔적을 기록으로 남깁니다.
        
        
4. **과제 태도**
    - 수업에서 다루지 않았더라도, 문제 해결에 필요하다고 판단한 방법은 충분히 탐색하고 시도해보세요.
    - 이번 과제에서는 **배운 내용만 반복하는 것보다, 필요한 내용을 스스로 찾아보고 적용해보는 태도**도 중요합니다.

Mission 0. 문제 해결 전략 수립 (Flow Planning)

1. 문제 정의
- 과제: 흉부 X-ray 영상을 정상과 폐렴으로 구분하는 이진 분류
- 임상적 의미: 폐렴을 놓치는 위음성이 정상을 폐렴으로 보는 위양성보다 위험하다. 그래서 정확도를 기준으로 성능을 평가하겠지만 **Recall**도 함께 확인한다.

2. 전체 흐름
    1) 데이터 이해(EDA)
    2) 실험 환경 고정 (seed, 검증 분할, 평가 지표)
    3) Baseline 구축 (CNN + 기본 설정)
    4) 한계 진단 (오분류 분석, 혼동행렬, Grad-CAM)
    5) 가설 수립
    6) 개선 실험 (한 번에 하나씩 바꾸기)
    7) 결과 비교와 해석 (성공, 실패 모두 기록)
    8) 최종 모델 선정
    9) 재현성 검증
    10) 제출

3. 단계별 계획
    1) 데이터 확인
        - 클래스 비율, 이미지 크기와 채널, 밝기 분포를 확인한다.
        - 정상과 폐렴 샘플을 확인한다.
    2) 재현성 확보
        - seed를 고정
        - 검증 데이터는 stratified 분할로 나누어 클래스 비율을 유지한다.
        - 평가지표는 Accuracy score를 주 지표로 쓰지만, AUC·F1·민감도·특이도를 보조지표로 함께 본다.
    3) Baseline
        - CNN을 활용한 Baseline 모델을 만든다.
    4) 전이 학습
        - ResNet50, 백본 고정, 분류층만 학습
    5) Fine tuning
    6) 한계 진단
        - 혼동행렬 사용
        - 학습 곡선으로 과적합 여부 확인
        - Grad-CAM으로 모델이 폐 영역을 보고 판단하는지 확인
    7) 가설 검증과 개선
    8) 최종 산출물

# 1. 데이터 확인 (train 데이터)

1) .csv 파일 
    - 데이터 크기: 5216개, 2개의 컬럼 (file_name, label)
    - 데이터 타입: file_name(str), label(int)
    - 결측치 없음
    - label: 0, 1 이진분류 (0: 정상, 1: 폐렴)
    - label: 0 - 1341개(26%), 1 - 3875개(74%) => 불균형 상태

2) .csv 파일, .png파일 정확히 매칭되어 있음

3) 데이터 확인 시사점
    - 두 가지 라벨 사이의 불균형으로 stratified 분할, 불균형 보정 필요함


In [ ]:
# 라이브러리 호출
from PIL import Image  # 이미지를 파이썬에서 처리하는 라이브러리
import pandas as pd    # 정형데이터 전처리 및 분석
import numpy as np     # 연속형 숫자 자료 처리
import os              # 파이썬이 파일 시스템에 접근하는 라이브러리
import matplotlib.pyplot as plt # 이미지와 그래프 시각화

In [12]:
# 경로 지정
BASE_DIR = BASE_DIR  = "/Users/mac/Desktop/deep_learning/chest_xray" # 각자 환경에 맞게 파일 경로지정
CSV_PATH = os.path.join(BASE_DIR, "train.csv")
TRAIN_DIR = os.path.join(BASE_DIR, "train")

# 경로 확인
print("CSV 존재:", os.path.exists(CSV_PATH))
print("이미지 폴더 존재:", os.path.exists(TRAIN_DIR))

CSV 존재: True
이미지 폴더 존재: True


In [ ]:
# 라벨 파일(csv) 형태 확인

df = pd.read_csv(CSV_PATH)

print("파일 확인:", df.shape) # 데이터 크기 확인
print(df.head(5))           # 5개의 행 출력해보기
df.info()                   # 데이터 정보

# 데이터 품질 확인
print("결측치:\n", df.isnull().sum()) 
print("중복 파일명:", df['file_name'].duplicated().sum())
print("라벨 종류:\n", df['label'].unique()) # 분류의 형태 확인 0, 1의 이진 분류인지 다중분류인지
print("라벨 분포:\n", df['label'].value_counts())
print("라벨 비율:\n", df['label'].value_counts(normalize=True).round(2)) # label 비율로 확인

# 출력 결과
# 데이터의 크기: 5216개, 2개의 컬럼(file_name, label)
# file_name: 데이터타입 str, label: 데이터 타입 int

# 결측치 없음
# 중복 파일 이름 없음
# 라벨 종류: 0, 1 이진 분류
# 라벨 분포: 0 - 1341개, 1 - 3875개 =>  불균형 상태
# 라벨 비율: 0 - 26%, 1- 74%



파일 확인: (5216, 2)
        file_name  label
0  train_0001.png      0
1  train_0002.png      0
2  train_0003.png      0
3  train_0004.png      0
4  train_0005.png      0
<class 'pandas.DataFrame'>
RangeIndex: 5216 entries, 0 to 5215
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   file_name  5216 non-null   str  
 1   label      5216 non-null   int64
dtypes: int64(1), str(1)
memory usage: 81.6 KB
결측치:
 file_name    0
label        0
dtype: int64
중복 파일명: 0
라벨 종류:
 [0 1]
라벨 분포:
 label
1    3875
0    1341
Name: count, dtype: int64
라벨 비율:
 label
1    0.74
0    0.26
Name: proportion, dtype: float64


In [ ]:
# 이미지 폴더 확인

# 이미지 파일이름 리스트로 변환
file_list = sorted([f for f in os.listdir(TRAIN_DIR) if f.endswith(".png")])

print("이미지 파일 수:", len(file_list))

# CSV와 이미지 폴더와 데이터 일치하는지 확인
print("CSV에는 있으나 이미지 폴더에 없는 파일:", len(set(df['file_name']) - set(file_list)))
print("이미지 폴더에는 있으나 CSV에는 없는 파일:", len(set(file_list)) - len(set(df['file_name'])))

# 출력 결과: csv파일 - 이미지 파일 매칭 잘되어 있음

이미지 파일 수: 5216
CSV에는 있으나 이미지 폴더에 없는 파일: 0
이미지 폴더에는 있으나 CSV에는 없는 파일: 0
